# Video Game Sales Prediction Using Historical Game and Review Data

**Goal:** predict `Global_Sales` (in millions) using historical game metadata and review information.

This notebook is organized from data loading through cleaning, EDA, feature preparation, model training, evaluation, and interpretation. **Preprocessing is performed manually** with Pandas; this notebook does not use `Pipeline` or `ColumnTransformer`.

> Put `Video_Games_Sales_as_at_22_Dec_2016.csv` in the same folder as this notebook, or update `DATA_PATH` in the next cell.

In [ ]:
# 1. Import libraries and load data
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("Video_Games_Sales_as_at_22_Dec_2016.csv")
if not DATA_PATH.exists():
    candidates = list(Path.cwd().glob("*.csv"))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            "CSV not found. Put Video_Games_Sales_as_at_22_Dec_2016.csv "
            "beside this notebook or set DATA_PATH to its location."
        )

df = pd.read_csv(DATA_PATH)
print("Loaded file:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Initial data inspection

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nNumerical summary:")
display(df.describe(include="all").T)
print("\nMissing values:")
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Percentage", ascending=False)
display(missing_summary[missing_summary["Missing_Count"] > 0])
print("Duplicate rows:", df.duplicated().sum())

## 3. Data cleaning and validation

- Remove exact duplicate rows.
- Convert score, count, year, and sales columns to numeric values.
- Values such as `tbd` in `User_Score` become missing values.
- Drop rows with a missing target (`Global_Sales`) or game name.
- Keep missing feature values for imputation after the train-test split.
- Do not remove high-sales outliers automatically: blockbuster sales may be legitimate.

In [ ]:
df = df.copy()

# Remove exact duplicate records
duplicate_count = int(df.duplicated().sum())
if duplicate_count:
    df = df.drop_duplicates().copy()
print("Duplicate rows removed:", duplicate_count)

# Convert columns to appropriate numeric types when present
numeric_candidates = [
    "Year_of_Release", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales",
    "Global_Sales", "Critic_Score", "Critic_Count", "User_Score", "User_Count"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalize whitespace in categorical text fields
categorical_candidates = ["Name", "Platform", "Genre", "Publisher", "Developer", "Rating"]
for col in categorical_candidates:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace("", pd.NA)

# Target and name are essential for this task
if "Global_Sales" not in df.columns:
    raise KeyError("Expected target column 'Global_Sales' was not found.")
rows_before = len(df)
df = df.dropna(subset=["Global_Sales"]).copy()
if "Name" in df.columns:
    df = df.dropna(subset=["Name"]).copy()
print("Rows removed for missing target/name:", rows_before - len(df))

# Basic domain checks: report suspicious values before deciding whether to remove them
sales_cols = [c for c in ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"] if c in df.columns]
print("Negative sales values by column:")
for col in sales_cols:
    print(f"{col}: {(df[col] < 0).sum()}")
if "Critic_Score" in df.columns:
    print("Critic scores outside 0-100:", ((df["Critic_Score"] < 0) | (df["Critic_Score"] > 100)).sum())
if "User_Score" in df.columns:
    print("User scores outside 0-10:", ((df["User_Score"] < 0) | (df["User_Score"] > 10)).sum())
print("Cleaned dataset shape:", df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing_Count").head(15))

## 4. Exploratory data analysis (EDA)

In [ ]:
# Numeric distributions
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_cols].hist(figsize=(16, 12), bins=35)
plt.suptitle("Distributions of Numeric Features", y=1.02)
plt.tight_layout()
plt.show()

# Global sales is usually right-skewed; inspect both raw and log scale
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["Global_Sales"], bins=50, ax=axes[0])
axes[0].set_title("Global Sales Distribution")
axes[0].set_xlabel("Global Sales (millions)")
sns.histplot(np.log1p(df["Global_Sales"].clip(lower=0)), bins=50, ax=axes[1])
axes[1].set_title("Log1p Global Sales Distribution")
axes[1].set_xlabel("log1p(Global Sales)")
plt.tight_layout()
plt.show()

In [ ]:
# Top-selling games
show_cols = [c for c in ["Name", "Platform", "Year_of_Release", "Genre", "Global_Sales"] if c in df.columns]
display(df.nlargest(10, "Global_Sales")[show_cols])

# Most represented platforms and genres
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if "Platform" in df.columns:
    df["Platform"].value_counts().head(10).sort_values().plot(kind="barh", ax=axes[0])
    axes[0].set_title("Top 10 Platforms by Game Count")
if "Genre" in df.columns:
    df["Genre"].value_counts().sort_values().plot(kind="barh", ax=axes[1])
    axes[1].set_title("Games by Genre")
plt.tight_layout()
plt.show()

In [ ]:
# Average global sales by genre and platform
if "Genre" in df.columns:
    genre_sales = df.groupby("Genre", dropna=False)["Global_Sales"].mean().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    genre_sales.plot(kind="bar")
    plt.title("Average Global Sales by Genre")
    plt.ylabel("Mean Global Sales (millions)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
if "Platform" in df.columns:
    platform_sales = df.groupby("Platform")["Global_Sales"].mean().sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 5))
    platform_sales.sort_values().plot(kind="barh")
    plt.title("Top 15 Platforms by Average Global Sales")
    plt.xlabel("Mean Global Sales (millions)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlations among numeric variables
corr_cols = [c for c in ["Global_Sales", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Year_of_Release", "Critic_Score", "Critic_Count", "User_Score", "User_Count"] if c in df.columns]
plt.figure(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix for Selected Numeric Columns")
plt.tight_layout()
plt.show()

# Check extreme values without deleting legitimate blockbusters
print("Highest global sales:")
display(df.nlargest(10, "Global_Sales")[show_cols])
for col in ["Global_Sales", "Critic_Score", "Critic_Count", "User_Count"]:
    if col in df.columns:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        n = int(((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum())
        print(f"{col}: {n} observations outside the 1.5*IQR range")

## 5. Define features and target

`Global_Sales` is the target. Regional sales (`NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`) are intentionally excluded because they are components of global sales and would leak the target. This setup is more appropriate when predicting sales from game metadata and review data. If the intended use case is after regional sales are already known, define that different prediction task explicitly.

In [ ]:
categorical_features = [c for c in ["Platform", "Genre", "Publisher", "Developer", "Rating"] if c in df.columns]
numerical_features = [c for c in ["Year_of_Release", "Critic_Score", "Critic_Count", "User_Score", "User_Count"] if c in df.columns]
feature_columns = categorical_features + numerical_features

if not feature_columns:
    raise ValueError("No expected feature columns were found. Check the dataset column names.")

X = df[feature_columns].copy()
y = df["Global_Sales"].copy()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)
print("X shape:", X.shape, "| y shape:", y.shape)
display(X.head())

## 6. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Manual preprocessing (no Pipeline)

Imputation values are learned from the training set only. Missing categories are represented as `Unknown`. One-hot encoding is performed with `pandas.get_dummies`, and the test columns are aligned to the training columns.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

# Numeric imputation: fill using training medians
for col in numerical_features:
    median_value = X_train[col].median()
    if pd.isna(median_value):
        median_value = 0
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

# Categorical imputation
for col in categorical_features:
    X_train[col] = X_train[col].astype("object").where(X_train[col].notna(), "Unknown")
    X_test[col] = X_test[col].astype("object").where(X_test[col].notna(), "Unknown")

# One-hot encode categorical features separately
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features, dtype=int)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_features, dtype=int)

# Ensure both sets have identical columns and order
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Missing values in encoded train:", int(X_train_encoded.isna().sum().sum()))
print("Missing values in encoded test:", int(X_test_encoded.isna().sum().sum()))
print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)
assert X_train_encoded.columns.equals(X_test_encoded.columns)

## 8. Train regression models and evaluate

In [ ]:
models = {
    "Baseline": DummyRegressor(strategy="mean"),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=15, min_samples_leaf=5, random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1
    )
}

results = []
trained_models = {}
predictions_by_model = {}

for name, model in models.items():
    print("Training", name)
    model.fit(X_train_encoded, y_train)
    predictions = model.predict(X_test_encoded)
    predictions_by_model[name] = predictions
    trained_models[name] = model
    mse = mean_squared_error(y_test, predictions)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.round(4))
print("Best model by test RMSE:", results_df.loc[0, "Model"])

In [ ]:
# Compare evaluation metrics
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(results_df["Model"], results_df["RMSE"])
axes[0].set_title("RMSE by Model (lower is better)")
axes[0].set_ylabel("RMSE")
axes[1].bar(results_df["Model"], results_df["R2"])
axes[1].set_title("R² by Model (higher is generally better)")
axes[1].set_ylabel("R²")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 9. Actual versus predicted sales

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
best_predictions = predictions_by_model[best_model_name]

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.35)
upper = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([0, upper], [0, upper], "k--", label="Perfect prediction")
plt.xlabel("Actual Global Sales (millions)")
plt.ylabel("Predicted Global Sales (millions)")
plt.title(f"Actual vs Predicted Global Sales — {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

print("Best model:", best_model_name)
print("MAE:", round(mean_absolute_error(y_test, best_predictions), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, best_predictions)), 4))
print("R²:", round(r2_score(y_test, best_predictions), 4))

## 10. Feature importance

In [ ]:
# Tree-based models provide feature importances. Use Random Forest if it was trained.
rf_model = trained_models.get("Random Forest")
if rf_model is not None:
    importance_df = pd.DataFrame({
        "Feature": X_train_encoded.columns,
        "Importance": rf_model.feature_importances_
    }).sort_values("Importance", ascending=False).reset_index(drop=True)
    display(importance_df.head(20))

    top_features = importance_df.head(15).sort_values("Importance")
    plt.figure(figsize=(10, 7))
    plt.barh(top_features["Feature"], top_features["Importance"])
    plt.xlabel("Feature Importance")
    plt.title("Top 15 Random Forest Features")
    plt.tight_layout()
    plt.show()
else:
    print("Random Forest was not trained; feature importance is unavailable.")

## 11. Conclusion and reporting notes

Use the actual `results_df` output to write your findings. Do not copy example metrics from earlier runs.

In your conclusion, report:
1. Which model performed best on the held-out test set and its MAE, RMSE, and R².
2. Which features had the highest Random Forest importance.
3. Important limitations: historical data may not represent current markets; missing scores are imputed; high-sales games are difficult to predict; and feature importance is not proof of causation.
4. The prediction target is global sales in millions. Regional sales were excluded from predictors to reduce target leakage.

**Reproducibility:** the split and models use `random_state=42`. Restart the kernel and run all cells from top to bottom before submitting.